# Prepare 2025 flight and weather data

Use shared flight and weather preparation modules with the fixed 2024 airport selection. The dataset is reserved for final evaluation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_flights import (
    COLUMNS, load_flights, prepare_timestamps, save_flights,
)

YEAR = 2025

## Load and validate flights

Read exactly twelve monthly files. Retain the selected origins, exclude cancelled flights and missing labels, then check dates, duplicate flight identifiers, and label consistency.

In [ ]:
flights, audit = load_flights(PROJECT_ROOT, YEAR)
missingness = flights[COLUMNS].isna().sum().rename("missing")

display(audit)
display(missingness)
print(f"Retained before timestamp checks: {len(flights):,} flights")

## Prepare UTC timestamps

Convert airport-local scheduled departures to UTC. A recorded time of 2400 means the following midnight. Exclude ambiguous or nonexistent daylight-saving times and retain them in an audit. Calculate prediction cutoffs two elapsed hours before departure.

In [ ]:
flights, excluded, timestamp_audit = prepare_timestamps(flights)

display(timestamp_audit)
display(excluded[["FlightDate", "Origin", "Dest", "CRSDepTime",
                  "scheduled_departure_local", "timezone"]])

## Save flight dataset and audits

The missingness audit describes retained flights before timestamp exclusions. Monthly cleaning counts and timestamp exclusions are recorded separately. The historical 2024 `excluded_ambiguous_departures.csv` is preserved; new runs use `excluded_unresolved_departures.csv` to cover both ambiguous and nonexistent times.

In [ ]:
save_flights(
    PROJECT_ROOT, YEAR, flights, audit, missingness, excluded, timestamp_audit
)
print(f"Saved {len(flights):,} flights for {YEAR}")

## Prepare weather matches

Use the shared NOAA quality rules, retain nullable trace/unlimited indicators, and match only observations at or before the two-hour prediction cutoff, with a maximum age of 60 minutes. January 1 of the following UTC year covers late local-year departures. All flights remain, including unmatched records. Observation time is used as a proxy for availability; reporting latency is not modelled. Only temperature, dew point, wind speed, and visibility are currently model predictors.

In [ ]:
from src.prepare_weather import prepare_weather, save_weather

combined, coverage, weather_missingness, quality_counts, stations = (
    prepare_weather(PROJECT_ROOT, YEAR)
)
display(coverage.round(2))
display(weather_missingness.round(2))

## Save weather dataset and audits

In [ ]:
save_weather(
    PROJECT_ROOT, YEAR, combined, coverage, weather_missingness,
    quality_counts, stations,
)
print(f"Saved {len(combined):,} flights with weather for {YEAR}")